In [2]:
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel
from scipy.optimize import minimize
from scipy.stats import norm
import warnings

warnings.filterwarnings('ignore')

In [3]:
def ucb_acquisition(x, gp, Y_max, beta):
    #Upper Confidence Bound (UCB) acquisition function
    x = x.reshape(1, -1)

    # Get the mean (mu) and std (sigma) from the GP
    mu, sigma = gp.predict(x, return_std=True)
    
    # Calculate UCB: mu + sqrt(beta) * sigma
    ucb = mu + np.sqrt(beta) * sigma
    
    return -ucb[0] # Return the negative UCB for minimisation

In [4]:
submission_queries = {} # Dictionary to hold all 8 query strings

base_directory = '..\..\data\initial_data'
beta_exploration = 2.0 #will increase for functions 6 to 8

print(f"\n--- Function-01 ---")
X = np.load('initial_inputs.npy')
Y = np.load('initial_outputs.npy')

# dimensionality
dimension = X.shape[1]

if Y.ndim > 1:
    Y = Y.flatten()
    
Y_max = np.max(Y)


print(f"{dimension}-Dimension - Y_max: {Y_max:.4f}")

# 2. Build and Train GP
kernel = Matern(length_scale=np.ones(dimension), length_scale_bounds=(1e-1, 1e2), nu=2.5)
gp = GaussianProcessRegressor(
    kernel=kernel, alpha=1e-6, n_restarts_optimizer=20, random_state=42)
gp.fit(X, Y)

# 3. Define the Optimisation Task
# Bounds for all functions are assumed to be [0.0, 1.0] for all dimensions.
bounds = [(0.0, 1.0)] * dimension

# Set the UCB exploration parameter based on dimensionality (Strategy: High Beta)
# Increase Beta for higher dimensions where data is sparser.
beta = beta_exploration
if dimension >= 5:
    beta = beta_exploration * 1.5 # More aggressive exploration for high-D
    
# The objective function (to be minimse) is the negative UCB
ucb_objective = lambda x: ucb_acquisition(x, gp, Y_max, beta)

# 4. Find the Argmax (i.e., minimise the negative UCB)
# Use multiple random starts to avoid local minima in the acquisition function
best_ucb_value = np.inf

x_next = None

for _ in range(20*dimension): # 20*D random restarts for better robustness
    x0 = np.random.uniform(0, 1, dimension)
    
    # Use a general minimiser (L-BFGS-B is good for bounded problems)
    res = minimize(ucb_objective, x0, bounds=bounds, method='L-BFGS-B')
    
    if res.fun < best_ucb_value:
        best_ucb_value = res.fun
        x_next = res.x



--- Function-01 ---
2-Dimension - Y_max: 0.0000
